# [5.4] Mamba State Tracking - Exercises

The previous section made the Mamba execution paths agree. This section asks a mechanistic question: when a recurrent architecture solves a stateful task, can we find the state in hidden activations, and can we intervene on it?

<img src="../../instructions/assets/mamba_state_tracking_ladder.svg" width="820">

The ladder is deliberately stricter than "fit a probe":

1. generate tasks where the true latent state is known,
2. fit probes on early positions and test on later positions,
3. intervene along a probe-derived state direction,
4. compare against random-direction and random-label controls,
5. train a tiny Mamba organism and a tiny Transformer baseline on the same task,
6. verify official Mamba hidden-state extraction on CUDA.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
from torch import nn
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part4_mamba_state_tracking"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_mamba_state_tracking.tests as tests
import part4_mamba_state_tracking.utils as utils
from arena_ext.mamba import MambaConfig, TinyMambaModel

MAIN = __name__ == "__main__"


@dataclass(frozen=True)
class StateTrackingBatch:
    tokens: t.Tensor
    states: t.Tensor
    task: str
    vocab: dict[int, str]


@dataclass(frozen=True)
class LinearProbe:
    weight: t.Tensor
    bias: t.Tensor


@dataclass(frozen=True)
class ProbeReport:
    train_accuracy: float
    test_accuracy: float
    num_train: int
    num_test: int


@dataclass(frozen=True)
class InterventionReport:
    source_prediction: int
    target_prediction: int
    intervened_prediction: int
    target_logit_delta: float
    passed: bool


## 1. Synthetic State-Tracking Tasks

Start with tasks where the latent state is exactly known at every token. Parity is cumulative XOR; bracket depth is a bounded stack depth after each open or close action.

<details><summary>Expected output</summary>

The tests should print:

```text
All tests in `test_generate_parity_task_matches_cumulative_xor` passed!
All tests in `test_generate_bracket_depth_task_is_bounded_and_consistent` passed!
```

</details>

<details><summary>Help - observable tokens vs latent state</summary>

The tokens are the input. The state is the running variable a model has to maintain. For parity, the state can change every time a `1` appears. For bracket depth, the state changes by +1 or -1 but is constrained by the generator to stay in `[0, max_depth]`.

</details>

<details><summary>Solution</summary>

For parity, use `tokens.cumsum(dim=-1) % 2`. For bracket depth, sample a proposed open/close action, force opens at depth zero and closes at `max_depth`, then update depth and store it at every position.

</details>


In [ ]:
def generate_parity_task(batch: int, seq_len: int, seed: int = 0) -> StateTrackingBatch:
    """Generate binary-token sequences labelled by cumulative XOR."""
    raise NotImplementedError()


def generate_bracket_depth_task(
    batch: int,
    seq_len: int,
    *,
    max_depth: int = 4,
    seed: int = 0,
) -> StateTrackingBatch:
    """Generate bracket actions labelled by bounded nonnegative stack depth."""
    raise NotImplementedError()


tests.test_generate_parity_task_matches_cumulative_xor(generate_parity_task)
tests.test_generate_bracket_depth_task_is_bounded_and_consistent(generate_bracket_depth_task)


## 2. Synthetic Hidden States and Held-Out Positions

Before probing a trained model, debug the probe pipeline on a control where the hidden state literally contains the answer: a one-hot vector for the latent state, optionally with small noise.

<details><summary>Expected output</summary>

The tests should print:

```text
All tests in `test_one_hot_state_features_shape_noise_and_reference` passed!
All tests in `test_make_position_split_masks_are_ordered_and_disjoint` passed!
```

</details>

<details><summary>Help - why split by position?</summary>

A random token split would let train and test see the same position distribution. Here we want to know whether a probe trained on early positions reads the same state variable at later positions, so the split is early positions vs later positions.

</details>

<details><summary>Solution</summary>

Use `F.one_hot(states, num_classes=num_states).float()` and deterministic `torch.Generator` noise. Build masks from `torch.arange(seq_len) < split` and expand them to match the batch.

</details>


In [ ]:
def one_hot_state_features(
    states: t.Tensor,
    num_states: int | None = None,
    *,
    noise_scale: float = 0.0,
    seed: int = 0,
) -> t.Tensor:
    """Represent each latent state by a noisy one-hot hidden vector."""
    raise NotImplementedError()


def make_position_split(
    states: t.Tensor,
    *,
    train_fraction: float = 0.5,
) -> tuple[t.Tensor, t.Tensor]:
    """Split every sequence into early train positions and later test positions."""
    raise NotImplementedError()


tests.test_one_hot_state_features_shape_noise_and_reference(one_hot_state_features)
tests.test_make_position_split_masks_are_ordered_and_disjoint(make_position_split)


## 3. Closed-Form Linear Probe

Fit a ridge-regression probe from hidden states to latent-state labels. In the one-hot control, the probe should recover all states and generalize to later positions.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_fit_linear_probe_recovers_held_out_one_hot_states` passed!
```

The held-out-position test accuracy should be above `0.99` in this control.

</details>

<details><summary>Help - probe accuracy is not causality</summary>

A high-accuracy probe says the state is linearly accessible. It does not prove the model uses that direction. That is why the next exercise uses interventions and random-direction controls.

</details>

<details><summary>Solution</summary>

Flatten `(batch, seq)` into one example axis, optionally mask it, one-hot the labels, append a bias column to the features, then solve the ridge-regularized normal equation.

</details>


In [ ]:
def _flatten_masked(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    mask: t.Tensor | None,
) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


def fit_linear_probe(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    *,
    num_classes: int | None = None,
    train_mask: t.Tensor | None = None,
    ridge: float = 1e-3,
) -> LinearProbe:
    """Fit a closed-form ridge-regression probe from hidden states to labels."""
    raise NotImplementedError()


def probe_logits(hidden_states: t.Tensor, probe: LinearProbe) -> t.Tensor:
    raise NotImplementedError()


def probe_predictions(hidden_states: t.Tensor, probe: LinearProbe) -> t.Tensor:
    raise NotImplementedError()


def probe_accuracy(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    probe: LinearProbe,
    mask: t.Tensor | None = None,
) -> float:
    raise NotImplementedError()


def evaluate_probe_generalization(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    probe: LinearProbe,
    train_mask: t.Tensor,
    test_mask: t.Tensor,
) -> ProbeReport:
    """Evaluate a probe on train positions and held-out later positions."""
    raise NotImplementedError()


tests.test_fit_linear_probe_recovers_held_out_one_hot_states(
    fit_linear_probe,
    one_hot_state_features,
    make_position_split,
    evaluate_probe_generalization,
    probe_predictions,
)


## 4. Probe-Derived Causal Intervention

A stronger state claim is intervention: move a hidden vector from a source-state direction toward a target-state direction, then check whether the decoded state flips. A matched random direction should not do the same thing.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_probe_intervention_flips_decoded_state_with_random_control` passed!
```

</details>

<details><summary>Help - target minus source</summary>

Use the target probe vector minus the source probe vector. Adding only the target vector may increase the target logit, but it need not increase it relative to the source state.

</details>

<details><summary>Solution</summary>

The direction is `probe.weight[:, target_state] - probe.weight[:, source_state]`. For the random control, sample a random vector, normalize it, scale it to the norm of the target probe vector, and decode after adding it.

</details>


In [ ]:
def state_intervention_direction(
    probe: LinearProbe,
    source_state: int,
    target_state: int,
) -> t.Tensor:
    """Return the target-minus-source probe direction."""
    raise NotImplementedError()


def apply_state_intervention(
    hidden_state: t.Tensor,
    probe: LinearProbe,
    *,
    source_state: int,
    target_state: int,
    coefficient: float = 1.0,
) -> t.Tensor:
    """Move a hidden state along a probe-derived target direction."""
    raise NotImplementedError()


def intervention_report(
    hidden_state: t.Tensor,
    probe: LinearProbe,
    *,
    source_state: int,
    target_state: int,
    coefficient: float = 1.0,
) -> InterventionReport:
    """Check whether a probe-derived intervention flips the decoded state."""
    raise NotImplementedError()


def random_direction_control(
    hidden_state: t.Tensor,
    probe: LinearProbe,
    *,
    target_state: int,
    coefficient: float = 1.0,
    seed: int = 0,
) -> int:
    """Apply a random direction with matched norm and decode the new state."""
    raise NotImplementedError()


tests.test_probe_intervention_flips_decoded_state_with_random_control(
    intervention_report,
    random_direction_control,
)


## 5. Tiny Mamba State Classifier

The full CUDA path trains this classifier on generated bracket-depth data. In the notebook, first make the wrapper expose token-level logits and token-level hidden states, because probes and interventions need one hidden vector per position.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_tiny_mamba_state_classifier_forward_shapes` passed!
```

The logits should have shape `(batch, seq, num_states)`.

</details>

<details><summary>Help - final-token classifiers are not enough</summary>

A state-tracking model organism needs a label at every position. Returning only the last hidden state would be enough for sequence classification, but it destroys the probe surface for earlier states.

</details>

<details><summary>Solution</summary>

Instantiate the `TinyMambaModel` from section 5.3 with a small config, call `self.backbone(input_ids)` in `encode`, apply a linear head to every token, and optionally return the hidden states alongside logits.

</details>


In [ ]:
class TinyMambaStateClassifier(nn.Module):
    """Tiny supervised Mamba organism for bracket-depth state tracking."""

    def __init__(self, num_states: int = 4):
        super().__init__()
        config = MambaConfig(
            vocab_size=2,
            d_model=32,
            d_inner=64,
            d_state=8,
            d_conv=3,
            dt_rank=4,
            num_layers=1,
            tie_word_embeddings=False,
        )
        self.backbone = TinyMambaModel(config)
        self.head = nn.Linear(config.d_model, num_states)

    def encode(self, input_ids: t.Tensor) -> t.Tensor:
        raise NotImplementedError()

    def forward(
        self,
        input_ids: t.Tensor,
        *,
        return_hidden_states: bool = False,
    ) -> t.Tensor | tuple[t.Tensor, t.Tensor]:
        raise NotImplementedError()


tests.test_tiny_mamba_state_classifier_forward_shapes(TinyMambaStateClassifier)


## 6. Whole-Notebook Contract

Run this after the previous exercises pass. It composes the generated tasks, probe, intervention, and parity controls into the section's small CPU contract.

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Help - what this does not prove</summary>

This smoke test proves the analysis pipeline on generated and one-hot controls. It does not prove a trained Mamba learned the state. That claim lives in the CUDA report summarized below.

</details>


In [ ]:
# Run after implementing the previous cells.
tests.test_notebook_contract()


## Signature Result

The committed CUDA report trains a tiny Mamba and a tiny causal Transformer on the same generated bracket-depth task, checks random-label failure, performs learned hidden-state interventions, and loads an official Mamba checkpoint for hidden-state extraction.

| Check | Current result | Acceptance rule |
|---|---:|---|
| Tiny Mamba short accuracy | `0.974` | at least `0.90` |
| Tiny Mamba long accuracy | `0.932` | at least `0.85` |
| Tiny Mamba long late-position accuracy | `0.899` | at least `0.80` |
| Random-label Mamba long accuracy | `0.355` | at most `0.55` |
| Tiny Transformer long accuracy | `0.622` | Mamba beats by at least `0.20` |
| Learned intervention success rate | `0.989` | at least `0.90` |
| Random-direction target rate | `0.063` | at most `0.50` |
| Learned hidden probe test accuracy | `0.850` | at least `0.80` |
| Official hidden shape | `[4, 11, 768]` | exact shape |
| Peak VRAM | `0.324 GB` | below `24 GB` |

<details><summary>Interpreting the signature result</summary>

The result supports a local model-organism claim: a tiny Mamba learns the generated bracket-depth state and generalizes better than the trained tiny Transformer baseline under this setup; a random-label control fails; and learned hidden-state directions causally shift decoded state more than matched random directions. It does not say pretrained Mamba-130M solves bracket depth.

</details>

<details><summary>Help - why the Transformer baseline is here</summary>

The baseline is not meant to prove Mamba is universally better. It checks whether this generated setup actually stresses long-sequence recurrent state, rather than being solved equally well by any small causal architecture trained on the same short sequences.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return {
        "device": gpu["device"],
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "tiny_mamba_short_accuracy": gpu["tiny_mamba_short_accuracy"],
        "tiny_mamba_long_accuracy": gpu["tiny_mamba_long_accuracy"],
        "tiny_mamba_long_late_accuracy": gpu["tiny_mamba_long_late_accuracy"],
        "tiny_mamba_random_label_long_accuracy": gpu["tiny_mamba_random_label_long_accuracy"],
        "tiny_transformer_long_accuracy": gpu["tiny_transformer_long_accuracy"],
        "learned_state_intervention_success_rate": gpu["learned_state_intervention_success_rate"],
        "learned_state_intervention_random_target_rate": gpu["learned_state_intervention_random_target_rate"],
        "learned_hidden_probe_test_accuracy": gpu["learned_hidden_probe_test_accuracy"],
        "official_mamba_hidden_shape": gpu["official_mamba_hidden_shape"],
        "official_mamba_fast_kernel_available": gpu["official_mamba_fast_kernel_available"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    }


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_gpu_test()


## Limitations

This section is a generated-task model-organism lab. It does not claim pretrained Mamba-130M performs bracket-depth tracking, does not benchmark general long-context ability, and does not show that every hidden-state direction is causal. The official checkpoint path verifies hidden-state extraction and fast-kernel availability only. Broader state-tracking claims need more tasks, seeds, architectures, and held-out distributions.
